# Consultas de verificación

En este notebook se realizan consultas SQL sobre las tablas cargadas en BigQuery para verificar la integridad de los datos, comprobar las relaciones entre las entidades y validar que el modelo permite obtener información relevante para el análisis del negocio.

In [1]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

client = bigquery.Client(project=PROJECT_ID)

print(f"Conectado al proyecto: {client.project}")
print(f"Dataset: {DATASET_ID}")

Conectado al proyecto: project-sql-big-query-epm
Dataset: kte_ecom


## Verificación del número de registros

Se comprueba el número de registros almacenados en cada tabla para verificar que la carga de datos se ha realizado correctamente.

In [2]:
query = f"""
SELECT 'customers' AS tabla, COUNT(*) AS registros
FROM `{PROJECT_ID}.{DATASET_ID}.customers`

UNION ALL

SELECT 'categories', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.categories`

UNION ALL

SELECT 'products', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.products`

UNION ALL

SELECT 'orders', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`

UNION ALL

SELECT 'order_items', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.order_items`

UNION ALL

SELECT 'payments', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.payments`

UNION ALL

SELECT 'reviews', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.reviews`
"""

df_counts = client.query(query).to_dataframe()

df_counts

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tabla,registros
0,customers,500
1,categories,5
2,products,100
3,orders,2000
4,order_items,4387
5,payments,2000
6,reviews,504


## Verificación de identificadores duplicados

Se comprueba que los identificadores principales de cada tabla no estén repetidos, validando así la unicidad lógica de los registros.

In [3]:
query = f"""
SELECT 'customers' AS tabla, COUNT(*) AS duplicados
FROM (
    SELECT customer_id
    FROM `{PROJECT_ID}.{DATASET_ID}.customers`
    GROUP BY customer_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'categories', COUNT(*)
FROM (
    SELECT category_id
    FROM `{PROJECT_ID}.{DATASET_ID}.categories`
    GROUP BY category_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'products', COUNT(*)
FROM (
    SELECT product_id
    FROM `{PROJECT_ID}.{DATASET_ID}.products`
    GROUP BY product_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'orders', COUNT(*)
FROM (
    SELECT order_id
    FROM `{PROJECT_ID}.{DATASET_ID}.orders`
    GROUP BY order_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'order_items', COUNT(*)
FROM (
    SELECT order_item_id
    FROM `{PROJECT_ID}.{DATASET_ID}.order_items`
    GROUP BY order_item_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'payments', COUNT(*)
FROM (
    SELECT payment_id
    FROM `{PROJECT_ID}.{DATASET_ID}.payments`
    GROUP BY payment_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'reviews', COUNT(*)
FROM (
    SELECT review_id
    FROM `{PROJECT_ID}.{DATASET_ID}.reviews`
    GROUP BY review_id
    HAVING COUNT(*) > 1
)
"""

df_duplicates = client.query(query).to_dataframe()

df_duplicates

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tabla,duplicados
0,customers,0
1,categories,0
2,products,0
3,orders,0
4,order_items,0
5,payments,0
6,reviews,0


## Verificación de relaciones entre tablas

Se comprueba que las claves foráneas lógicas de cada tabla hagan referencia a registros existentes en las tablas relacionadas.

In [4]:
query = f"""
SELECT 'products -> categories' AS relacion, COUNT(*) AS errores
FROM `{PROJECT_ID}.{DATASET_ID}.products` p
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.categories` c
    ON p.category_id = c.category_id
WHERE c.category_id IS NULL

UNION ALL

SELECT 'orders -> customers', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders` o
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.customers` c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'order_items -> orders', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` oi
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
    ON oi.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT 'order_items -> products', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` oi
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.products` p
    ON oi.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 'payments -> orders', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.payments` p
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o
    ON p.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT 'reviews -> order_items', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.reviews` r
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` oi
    ON r.order_item_id = oi.order_item_id
WHERE oi.order_item_id IS NULL
"""

df_relations = client.query(query).to_dataframe()

df_relations

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,relacion,errores
0,products -> categories,0
1,orders -> customers,0
2,order_items -> orders,0
3,order_items -> products,0
4,payments -> orders,0
5,reviews -> order_items,0


## Verificación de reglas de negocio

Se comprueba que determinados campos respeten los rangos y condiciones definidos por el modelo, como las valoraciones, cantidades, descuentos, precios y stock.

In [5]:
query = f"""
SELECT 'reviews_rating' AS comprobacion, COUNT(*) AS errores
FROM `{PROJECT_ID}.{DATASET_ID}.reviews`
WHERE rating < 1 OR rating > 5

UNION ALL

SELECT 'order_items_quantity', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.order_items`
WHERE quantity <= 0

UNION ALL

SELECT 'order_items_discount', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.order_items`
WHERE discount < 0 OR discount > 1

UNION ALL

SELECT 'products_sale_price', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.products`
WHERE sale_price <= 0

UNION ALL

SELECT 'products_cost_price', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.products`
WHERE cost_price <= 0

UNION ALL

SELECT 'products_stock', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.products`
WHERE stock < 0

UNION ALL

SELECT 'payments_amount', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.payments`
WHERE amount <= 0
"""

df_business_rules = client.query(query).to_dataframe()

df_business_rules

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,comprobacion,errores
0,reviews_rating,0
1,order_items_quantity,0
2,order_items_discount,0
3,products_sale_price,0
4,products_cost_price,0
5,products_stock,0
6,payments_amount,0


## Verificación de coherencia entre estados y fechas

Se valida que las fechas de envío y entrega sean coherentes con el estado de cada pedido y que respeten el orden cronológico esperado.

In [6]:
query = f"""
SELECT 'shipping_before_order' AS comprobacion, COUNT(*) AS errores
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE shipping_date IS NOT NULL
  AND shipping_date < order_date

UNION ALL

SELECT 'delivery_before_shipping', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE delivery_date IS NOT NULL
  AND shipping_date IS NOT NULL
  AND delivery_date < shipping_date

UNION ALL

SELECT 'pending_with_shipping_date', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status = 'pending'
  AND shipping_date IS NOT NULL

UNION ALL

SELECT 'confirmed_with_shipping_date', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status = 'confirmed'
  AND shipping_date IS NOT NULL

UNION ALL

SELECT 'cancelled_with_shipping_date', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status = 'cancelled'
  AND shipping_date IS NOT NULL

UNION ALL

SELECT 'shipped_without_shipping_date', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status = 'shipped'
  AND shipping_date IS NULL

UNION ALL

SELECT 'delivered_without_delivery_date', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status = 'delivered'
  AND delivery_date IS NULL

UNION ALL

SELECT 'returned_without_delivery_date', COUNT(*)
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status = 'returned'
  AND delivery_date IS NULL
"""

df_dates_status = client.query(query).to_dataframe()

df_dates_status

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,comprobacion,errores
0,shipping_before_order,0
1,delivery_before_shipping,0
2,pending_with_shipping_date,0
3,confirmed_with_shipping_date,0
4,cancelled_with_shipping_date,0
5,shipped_without_shipping_date,0
6,delivered_without_delivery_date,0
7,returned_without_delivery_date,0


------------------------------------------------------------------------------------------------------------------------------------------------------

## Consultas analíticas

Una vez verificada la integridad y coherencia de los datos, se realizan consultas analíticas orientadas a obtener información relevante sobre la actividad del negocio.

Las consultas permiten analizar:

- Los ingresos obtenidos por mes.
- Los productos más vendidos.
- La distribución de clientes por país.
- El tiempo medio de entrega de los pedidos.
- Los ingresos generados por categoría de producto.

### Ingresos por mes

Se calculan los ingresos obtenidos cada mes a partir de los pagos completados, permitiendo observar la evolución temporal de las ventas.

In [7]:
query = f"""
SELECT
    DATE_TRUNC(o.order_date, MONTH) AS month,
    ROUND(SUM(p.amount), 2) AS revenue
FROM `{PROJECT_ID}.{DATASET_ID}.payments` AS p
JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON p.order_id = o.order_id
WHERE p.status = 'completed'
GROUP BY month
ORDER BY month
"""

df_monthly_revenue = client.query(query).to_dataframe()

df_monthly_revenue

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month,revenue
0,2023-10-01,8278.98
1,2023-11-01,4369.87
2,2023-12-01,19441.40
3,2024-01-01,6463.68
4,2024-02-01,16009.17
5,2024-03-01,18104.30
6,2024-04-01,25903.86
7,2024-05-01,13234.13
8,2024-06-01,28880.97
9,2024-07-01,43975.11


### Productos más vendidos

Se identifican los productos con mayor número de unidades vendidas, permitiendo conocer cuáles tienen una mayor demanda.

In [8]:
query = f"""
SELECT
    p.product_id,
    p.name AS product,
    SUM(oi.quantity) AS units_sold
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.orders` AS o
    ON oi.order_id = o.order_id
WHERE o.status NOT IN ('cancelled', 'returned')
GROUP BY p.product_id, p.name
ORDER BY units_sold DESC
LIMIT 10
"""

df_top_products = client.query(query).to_dataframe()

df_top_products

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,product,units_sold
0,23,VertexBook 23,84
1,95,PulseTab 95,82
2,70,EchoSound 70,81
3,48,NexusGear 48,79
4,64,NovaTab 64,78
5,30,NovaGear 30,78
6,40,NovaSound 40,76
7,37,OrbitBook 37,74
8,61,Nexus 61,73
9,41,PulseTab 41,73


### Clientes por país

Se analiza la distribución geográfica de los clientes, mostrando el número de clientes registrados en cada país.

In [9]:
query = f"""
SELECT
    country,
    COUNT(*) AS customers
FROM `{PROJECT_ID}.{DATASET_ID}.customers`
GROUP BY country
ORDER BY customers DESC, country
"""

df_customers_country = client.query(query).to_dataframe()

df_customers_country

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,country,customers
0,Fiji,7
1,Zimbabwe,7
2,Armenia,6
3,Chipre,6
4,Ecuador,6
...,...,...
171,Rumania,1
172,Santo Tomé y Príncipe,1
173,Sri Lanka,1
174,Swazilandia,1


### Tiempo medio de entrega

Se calcula el número medio de días transcurridos entre la realización del pedido y su entrega.

In [10]:
query = f"""
SELECT
    ROUND(
        AVG(DATE_DIFF(delivery_date, order_date, DAY)),
        2
    ) AS avg_delivery_days
FROM `{PROJECT_ID}.{DATASET_ID}.orders`
WHERE status IN ('delivered', 'returned')
    AND delivery_date IS NOT NULL
"""

df_avg_delivery = client.query(query).to_dataframe()

df_avg_delivery

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,avg_delivery_days
0,5.96


### Ingresos por categoría

Se calculan los ingresos generados por cada categoría a partir de las líneas de pedido asociadas a pagos completados.

In [11]:
query = f"""
SELECT
    c.category_id,
    c.name AS category,
    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - COALESCE(oi.discount, 0))
        ),
        2
    ) AS revenue
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories` AS c
    ON p.category_id = c.category_id
JOIN `{PROJECT_ID}.{DATASET_ID}.payments` AS pay
    ON oi.order_id = pay.order_id
WHERE pay.status = 'completed'
GROUP BY c.category_id, c.name
ORDER BY revenue DESC
"""

df_revenue_category = client.query(query).to_dataframe()

df_revenue_category

c:\Users\NitroPC\Pictures\KTE\TheBridge IA Engeneering\Docs\VSC\GitHubEPM\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_id,category,revenue
0,5,Accessories,733983.10
1,4,Tablets,693522.01
2,1,Smartphones,691241.95
3,3,Audio,685573.66
4,2,Laptops,676575.33
